In [1]:
'''
This is a notebook to help write config files for different arg inference tools
In cases where a large contig has to be broken down into multiple differing runs
'''

'\nThis is a notebook to help write config files for different arg inference tools\nIn cases where a large contig has to be broken down into multiple differing runs\n'

In [2]:
import yaml
import pandas as pd
import re
import glob
import numpy as np
import os

pd.set_option('display.max_colwidth', None)

In [3]:
#DEFINE Defaults:
MAX_CONTIG_LENGTH = 5_000_000
CONTIG_OVERLAP = 300_000
MIN_CONTIG_LENGTH = 750_000

NE = 10_000
SINGULARITY_IMAGE = "/grid/siepel/home/khalid/argsortium-inference/shared/container/arg_inference_tools.sif"
OUTPUT_DIR = "/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/singer"
CONFIG_OUTPUT = "/grid/siepel/home/khalid/argsortium-inference/singer/myconfigs/2026_05_11_runs"
input_folder_base = "/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09"


patterns = [
    f"{input_folder_base}/CEU_*_CHB_20_YRI_20/*.params.csv",
    f"{input_folder_base}/CEU_*_CHB_20_YRI_0/*.params.csv",
    f"{input_folder_base}/CEU_*_CHB_0_YRI_20/*.params.csv"
]

arg_inference_method = "singer".lower()

In [4]:
def sort_samples_str(samples_str):
    '''
    This function will be deprecated once the naming convention in the simulations has been updated.
    '''
    # Split into (pop, count) pairs: ["YRI", "20", "CEU", "0", "CHB", "0"]
    parts = samples_str.split("_")
    # Zip into pairs: [("YRI", "20"), ("CEU", "0"), ("CHB", "0")]
    pairs = [(parts[i], parts[i+1]) for i in range(0, len(parts), 2)]
    # Sort alphabetically by pop name and rejoin
    return "_".join([f"{pop}_{count}" for pop, count in sorted(pairs)])

def chunk_row(row):
    """Break a row into overlapping chunks if simulated_length > MAX_CONTIG_LENGTH.
    Pad chunks smaller than MIN_CONTIG_LENGTH by extending left or right."""
    if row["simulated_length"] <= MAX_CONTIG_LENGTH:
        chunks = [row.copy()]
        chunks[0]["inference_start"] = row["start"]
        chunks[0]["inference_end"] = row["end"]
        chunks[0]["inference_length"] = row["end"] - row["start"]
    else:
        step = MAX_CONTIG_LENGTH - CONTIG_OVERLAP
        chunks = []
        chunk_start = row["start"]

        while chunk_start < row["end"]:
            chunk_end = min(chunk_start + MAX_CONTIG_LENGTH, row["end"])
            new_row = row.copy()
            new_row["inference_start"] = chunk_start
            new_row["inference_end"] = chunk_end
            new_row["inference_length"] = chunk_end - chunk_start
            chunks.append(new_row)

            if chunk_end == row["end"]:
                break
            chunk_start += step

    # Pad any chunk below MIN_CONTIG_LENGTH
    for chunk in chunks:
        if chunk["inference_length"] < MIN_CONTIG_LENGTH:
            deficit = MIN_CONTIG_LENGTH - chunk["inference_length"]
            if chunk["inference_end"] == row["end"]:
                # At the end of the region — extend leftward
                chunk["inference_start"] = max(row["start"], chunk["inference_start"] - deficit)
            else:
                # At the start of the region — extend rightward
                chunk["inference_end"] = min(row["end"], chunk["inference_end"] + deficit)
            chunk["inference_length"] = chunk["inference_end"] - chunk["inference_start"]

    return chunks

In [7]:
#define all configs here
argweaver_config = {
    "singularity": SINGULARITY_IMAGE,
    "output_dir": "",
    "params_pattern": [],
    "max_contig_length": MAX_CONTIG_LENGTH,
    "start": None,
    "end": None,
    "Ne": None,
    "recomb_rate": None,
    "mcmc_samples": 4000, #defaults
    "compression": 10,
    "n_time_points": None,
    "sample_step": 20,
}

singer_config = {
    "singularity" : SINGULARITY_IMAGE,
    "output_dir" : "",
    "params_pattern" : [],
    "max_contig_length" : MAX_CONTIG_LENGTH,
    "start" : None,
    "end" : None,
    "Ne" : 2*10_000, #haploid Ne
    "mcmc_samples" : 2000, #SINGER converges quicker
    "recomb_ratio" : None,
    "thin" : 20,
    "polar" : 0.99 #use 0.99 for polarized data
    
}

relate_config = {
    
}

tsinfer_config = {
    
}

threads_config = {
    
}

asmc_clust_config = {
    
}

polegon_config = {
    
}

config_map = {
    "argweaver" : argweaver_config,
    "singer" : singer_config,
    "relate" : relate_config,
    "tsinfer" : tsinfer_config,
    "asmc_clust" : asmc_clust_config,
    "polegon_config" : polegon_config
}

config_to_use = config_map[arg_inference_method]

In [8]:
#find all params files and read them in to 1 pandas dataframe:
matched_params_files = sorted(
    f for pattern in patterns for f in glob.glob(pattern)
)

params_df = pd.concat(
    [pd.read_csv(f).assign(source_file=f) for f in matched_params_files],
    ignore_index=True
)
params_df["samples"] = params_df["samples"].apply(sort_samples_str)

In [9]:
grouped_df = params_df.groupby(["samples", "contig", "start", "end", "simulated_length"]).count()[["vcf_file"]]\
.rename(columns = {"vcf_file" : "number_of_seeds"}).reset_index()

chunked_df = pd.DataFrame(
    [chunk for row in grouped_df.itertuples(index=False)
     for chunk in chunk_row(row._asdict())]
).reset_index(drop=True)

chunked_df

,samples,contig,start,end,simulated_length,number_of_seeds,inference_start,inference_end,inference_length
0,CEU_0_CHB_0_YRI_20,chr1,55039445,60939445,5900000,10,55039445,60039445,5000000
1,CEU_0_CHB_0_YRI_20,chr1,55039445,60939445,5900000,10,59739445,60939445,1200000
2,CEU_0_CHB_0_YRI_20,chr11,116827019,120727019,3900000,10,116827019,120727019,3900000
3,CEU_0_CHB_0_YRI_20,chr14,20430758,105464069,85033311,10,20430758,25430758,5000000
4,CEU_0_CHB_0_YRI_20,chr14,20430758,105464069,85033311,10,25130758,30130758,5000000
...,...,...,...,...,...,...,...,...,...
70,CEU_0_CHB_20_YRI_20,chr14,20430758,105464069,85033311,10,100330758,105330758,5000000
71,CEU_0_CHB_20_YRI_20,chr14,20430758,105464069,85033311,10,104714069,105464069,750000
72,CEU_0_CHB_20_YRI_20,chr17,55039445,60939445,5900000,10,55039445,60039445,5000000
73,CEU_0_CHB_20_YRI_20,chr17,55039445,60939445,5900000,10,59739445,60939445,1200000


In [11]:
merged_file = pd.merge(chunked_df, params_df, on = ["contig", "samples", "start", "end", "simulated_length"])
merged_file["output_dir"] = merged_file[["samples", "contig", "start", "end"]]\
.apply(lambda x : "/".join([OUTPUT_DIR, str(x[0]), "_".join([str(x[1]), str(x[2]), str(x[3])])]), axis = 1)

/tmp/ipykernel_1682063/1326930485.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  .apply(lambda x : "/".join([OUTPUT_DIR, str(x[0]), "_".join([str(x[1]), str(x[2]), str(x[3])])]), axis = 1)


In [12]:
group_cols = ["samples", "contig", "start", "end", "inference_start", "inference_end", "inference_length", "number_of_seeds"]
output_configs = []
for group_vals, group in merged_file.groupby(group_cols):

    # Check 1: within each group, output_dir should be unique
    output_dirs = np.unique(group.loc[:,"output_dir"])
    assert len(output_dirs) == 1, f"Multiple output dirs for group: {group_vals}"

    # Check 2: across all groups, no two groups should share an output_dir
    output_dir_counts = merged_file.groupby("output_dir")[group_cols[0]].nunique()
    duplicates = output_dir_counts[output_dir_counts > 1]
    assert len(duplicates) == 0, f"output_dir shared across multiple groups:\n{duplicates}"

    param_files = list(np.unique(group.loc[:, "source_file"]))
    assert len(param_files) == group_vals[7], f"Number of seeds doesn't match unique param_file values {group_vals}\n{param_files}"

    config = config_to_use.copy()
    config["output_dir"] = output_dirs[0]
    config["params_pattern"] = param_files
    config["start"] = int(group_vals[4])
    config["end"] = int(group_vals[5])
    config = {k: v for k, v in config.items() if v is not None}

    # Write config — filename encodes samples, contig, and inference window
    samples, contig, *_, inf_start, inf_end, _ = group_vals
    config_filename = f"config.{samples}__{contig}_{inf_start}_{inf_end}.yaml"
    config_path = os.path.join(CONFIG_OUTPUT, config_filename)
    
    with open(config_path, "w") as f:
        yaml.dump(config, f, default_flow_style=False)

print(f"Wrote {len(merged_file.groupby(group_cols))} configs to {CONFIG_OUTPUT}")

Wrote 75 configs to /grid/siepel/home/khalid/argsortium-inference/singer/myconfigs/2026_05_11_runs
